# HADES: run one system

Builds one integrated HTC and anaerobic digestion system and computes its life cycle impacts, for any feedstock and set of operating conditions. The functional unit is the treatment of 1 t of wet feedstock.

- **Pathway A:** HTC of the feedstock, then anaerobic digestion of the process water.
- **Pathway B:** anaerobic digestion of the feedstock, then HTC of the digestate.

The settings follow the user decisions of the article (Supporting Information, Table S1): process configuration, feedstock properties, HTC conditions, AD conditions and LCA settings. The defaults are the Base scenario of the case study (food waste, 180 °C, 60 min, 30 d). Pathway A reproduces Paper 1 exactly. Pathway B uses the feedstock-specific digester feed of Paper 2 (for food waste: methane −13 %, hydrochar +21 % compared with Paper 1).

**Paper 2 options** (section 1, all off by default):
- **LCA context** (`CONTEXT`): Canadian province or territory, which sets the dataset locations and the industrial heat mix displaced by hydrochar (NRCan 2023).
- **Heat supply** (`HEAT_SUPPLY`): electricity, natural gas heat or the plant's own biogas, separately for the HTC reactor, the dryer and the digester.
- **Pathway B degradability** (`FEEDSTOCK_TYPE`, `B_FEED_OVERRIDES`): biodegradable fraction fD, soluble share and hydrolysis rate of the digester feed.
- **Pathway A process-water methane** (`PW_CH4_MULTIPLIER`): uncertainty of the process-water BMP, 0.55 to 1.45.

For many runs, use the sweep notebook instead.

**Before running:** create a Brightway project with ecoinvent and IMPACT World+ once with `python setup_hades.py` (see the README), then set `PROJECT` and `EI_NAME` below. The same model is available without code through the interface: `streamlit run app.py`.

## 1. Settings

In [ ]:
# =============================================================================
# 1. SETTINGS (user decisions, Supporting Information Table S1)
# =============================================================================

# ---- 1. Process configuration
PATHWAYS = ["A", "B"]          # "A", "B" or both

# ---- 2. Feedstock properties (% dry basis, O by difference; moisture % wet basis)
FEED = dict(WC=85.0, C_feed=38.25, H_feed=5.47, N_feed=3.66, S_feed=0.50, Ash_feed=9.10,
            VM_feed=82.6, FC_feed=8.3)                   # food waste (Base scenario)
FEEDSTOCK_TYPE = "food_waste"  # "food_waste", "ofmsw", "dairy_manure" or "sewage_sludge": Pathway B digester
                               # feed degradability (fD, soluble share, k_hyd) and digestate coefficients

# ---- 3. HTC operating conditions
T_HTC = 180                    # reactor temperature (C), 160 to 280
RT_HTC = 60                    # residence time (min)
TARGET_MOISTURE_HTC = 0.85     # moisture of the reactor feed (kg/kg); water is added if the feed is drier

# ---- 4. AD operating conditions
ADM1_MODE = "continuous"       # "continuous": full-scale digester at steady state (use for the LCA of a plant)
                               # "batch": BMP-test simulation (inoculum, blank-corrected), to compare with lab BMPs
BATCH_DAYS = 30                # batch test duration (d), batch mode only
HRT_DAYS = 30                  # hydraulic retention time (d)
COD_TARGET = 10.0              # COD of the diluted process water fed to the digester (kg COD/m3), Pathway A
TS_TARGET = 0.08               # total solids of the digester feed (kg/kg), Pathway B
T_SPINUP = 400                 # spin-up to steady state (d)
B_FEED_OVERRIDES = {}          # Pathway B: replace tabulated degradability, e.g. {"fD": 0.80, "f_sol": 0.30, "k_hyd": 0.20}
                               # (fD: biodegradable share of feed COD; f_sol: soluble share of it; k_hyd in 1/d)
PW_CH4_MULTIPLIER = 1.0        # Pathway A: multiplier on process-water methane, 1.0 = calibrated PW-ADM1
                               # (literature range 0.55 to 1.45; capped at 350 mL CH4/g COD fed)

# ---- Heat supply of the HTC reactor, the dryer and the digester
#      "electricity": grid electricity (HTC: electric heater, efficiency 0.9)
#      "ng"         : natural gas heat, condensing boiler >100 kW (delivered heat)
#      "own_biogas" : raw biogas burned on site before upgrading (less biomethane exported);
#                     biogas goes to the digester first, then HTC, then dryer; any shortfall from HEAT_BACKUP
#      Paper 1: htc="electricity", dryer="electricity", ad="ng"
HEAT_SUPPLY = dict(htc="electricity", dryer="electricity", ad="ng")
HEAT_BACKUP = "ng"             # source of an own-biogas shortfall: "ng" or "electricity"

# ---- 5. LCA settings
PROJECT = "my_hades_project"                 # Brightway project (created by setup_hades.py)
EI_NAME = "ecoinvent-3.12-cutoff"            # background database
CONTEXT = "QC_PAPER1"                        # LCA context preset (ADM1/lca_contexts.py): "QC_PAPER1" (Paper 1),
                                             # or a province/territory code, e.g. "QC", "ON", "AB", "NS" (NRCan 2023 heat mix)
HYDROCHAR_DISPLACES = "heat mix"             # "heat mix" (industrial heat mix) or "coal" (hard coal heat)
IW_MID = "IMPACT World+ Midpoint 2.2.1_regionalized for ecoinvent v3.12"
IW_DMG = "IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12"
METHODS = {"GWP100 (kg CO2 eq)":            (IW_MID, "Midpoint", "Climate change, short term"),
           "Human health (DALY)":           (IW_DMG, "Human health", "Total human health"),
           "Ecosystem quality (PDF.m2.yr)": (IW_DMG, "Ecosystem quality", "Total ecosystem quality")}

# ---- Advanced (defaults of the article)
MC_CHAR = 0.35                 # hydrochar moisture after the filter press (kg/kg)
MC_CHAR_DRY = 0.10             # hydrochar moisture after drying (kg/kg)
ISR_BATCH = 2.0                # inoculum to substrate ratio (VS basis), batch mode only

## 2. Load HADES and the background datasets

In [ ]:
# =============================================================================
# 2. HADES CODE AND BACKGROUND DATASETS
# =============================================================================
import os, sys, importlib.util
import pandas as pd
import bw2data as bd
from bw2calc import LCA

ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "ADM1")) else os.path.dirname(os.getcwd())   # repository root

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod; spec.loader.exec_module(mod)
    return mod

sb = load_module("system_builder", os.path.join(ROOT, "ADM1", "system_builder.py"))
lcc = load_module("lca_contributions", os.path.join(ROOT, "ADM1", "lca_contributions.py"))
lcx = load_module("lca_contexts", os.path.join(ROOT, "ADM1", "lca_contexts.py"))
bwl, hce = sb.bwl, sb.hce
ML_MODULE = os.path.join(ROOT, "ML training scripts and models", "htc_surrogate.py")   # HTC surrogate

# ---- input checks
O_feed = 100 - FEED["C_feed"] - FEED["H_feed"] - FEED["N_feed"] - FEED["S_feed"] - FEED["Ash_feed"]
if O_feed < 0:
    raise ValueError(f"C + H + N + S + ash = {100 - O_feed:.2f} % of dry matter (> 100 %)")
if abs(FEED["VM_feed"] + FEED["FC_feed"] + FEED["Ash_feed"] - 100) > 1.0:
    print(f"WARNING: VM + FC + ash = {FEED['VM_feed'] + FEED['FC_feed'] + FEED['Ash_feed']:.1f} % (expected 100 %)")
if not 160 <= T_HTC <= 280:
    raise ValueError(f"T_HTC = {T_HTC} C is outside 160-280 C (range of the PW-ADM1 S_I composition)")
feed = {**FEED, "O_feed": O_feed}
adm1_core = sb._adm1_core
if "B" in PATHWAYS:
    for name, table in (("degradability (ADM1.PATHWAY_B_FEED_PARAMS)", adm1_core.PATHWAY_B_FEED_PARAMS),
                        ("digestate coefficients (ad_characterization)", sb.ac.AD_CHARACTERIZATION)):
        if FEEDSTOCK_TYPE not in table:
            raise ValueError(f"FEEDSTOCK_TYPE {FEEDSTOCK_TYPE!r} has no {name}; known: {sorted(table)}")
    b_feed = adm1_core.pathway_b_feed_params(FEEDSTOCK_TYPE, B_FEED_OVERRIDES)   # checks the override keys
    for k, rng in (("fD", (0.05, 1.0)), ("f_sol", (0.0, 1.0)), ("k_hyd", (0.001, 20.0))):
        if not rng[0] <= b_feed[k] <= rng[1]:
            raise ValueError(f"B_FEED_OVERRIDES {k} = {b_feed[k]} outside {rng}")
    print(f"Pathway B feed ({FEEDSTOCK_TYPE}): fD {b_feed['fD']}, soluble share {b_feed['f_sol']}, "
          f"k_hyd {b_feed['k_hyd']} 1/d" + (f"  (overridden: {B_FEED_OVERRIDES})" if B_FEED_OVERRIDES else ""))
if ADM1_MODE not in ("continuous", "batch"):
    raise ValueError(f"ADM1_MODE must be 'continuous' or 'batch', got {ADM1_MODE!r}")
if ADM1_MODE == "batch":
    print(f"NOTE: batch mode simulates a {BATCH_DAYS}-day BMP test (ISR {ISR_BATCH}), not a running plant; "
          "use it to compare with lab BMP data, and continuous mode for the LCA of a plant.")
if PW_CH4_MULTIPLIER <= 0:
    raise ValueError("PW_CH4_MULTIPLIER must be > 0")
lo, hi_ = sb.PW_CH4_MULTIPLIER_RANGE
if not lo <= PW_CH4_MULTIPLIER <= hi_:
    print(f"WARNING: PW_CH4_MULTIPLIER = {PW_CH4_MULTIPLIER} outside the literature range {lo} to {hi_}")
for c, v in {**HEAT_SUPPLY, "backup": HEAT_BACKUP}.items():
    if c != "backup" and c not in ("htc", "dryer", "ad"):
        raise ValueError(f"HEAT_SUPPLY: unknown consumer {c!r} (use htc, dryer, ad)")
    if v not in ("electricity", "ng", "own_biogas") or (c == "backup" and v == "own_biogas"):
        raise ValueError(f"HEAT_SUPPLY / HEAT_BACKUP {c} = {v!r} not allowed")

# ---- LCA context: location priority and displaced heat mix
CTX = lcx.get_preset(CONTEXT)
LOC = CTX["location_priority"]               # dataset location priority (first match is used)
HEAT_MIX_SHARES = lcx.heat_mix_shares(CTX)
print(f"LCA context: {CTX['name']}  locations {LOC}")
print(f"  heat mix: {HEAT_MIX_SHARES}\n  source: {CTX['heat_mix_source']}" + (f"\n  note: {CTX['note']}" if CTX["note"] else ""))

# ---- Brightway project and background database
if PROJECT not in bd.projects:
    raise KeyError(f"Brightway project {PROJECT!r} not found. Run `python setup_hades.py --project {PROJECT}` first.")
bd.projects.set_current(PROJECT)
if EI_NAME not in bd.databases:
    raise KeyError(f"{EI_NAME!r} not in project {PROJECT!r}. Databases here: {list(bd.databases)}")
_deps = [d for d in bd.databases[EI_NAME].get("depends", []) if "biosphere" in d.lower() and "spatialized" not in d.lower()]
bwl.GENERIC_FLOW_DB = _deps[0] if _deps else "biosphere3"
ei, bios = bd.Database(EI_NAME), bd.Database(bwl.GENERIC_FLOW_DB)
for m in METHODS.values():
    if m not in bd.methods:
        raise KeyError(f"LCIA method not found in project {PROJECT!r}: {m}")

find = lambda q, prio=LOC, nf=None: bwl.find_process(ei, q, prio, name_filter=nf)
bg = dict(
    elec=find("market for electricity, medium voltage", nf="market for electricity, medium voltage"),
    coal_heat=find("heat production, at hard coal industrial furnace 1-10MW", nf="hard coal industrial furnace"),
    ad_heat=find("heat production, natural gas, at boiler condensing modulating >100kW", nf="heat production, natural gas, at boiler condensing modulating >100kW"),   # same boiler in every region
    ng_heat_market=find("market for heat, district or industrial, natural gas",
                        nf="market for heat, district or industrial, natural gas"),
    biomethane_heat_ref=find("heat production, biomethane, at boiler condensing modulating",
                             ["RoW", "GLO"] + LOC, nf="biomethane"),
    tap_water=find("market for tap water", nf="market for tap water"),
    wastewater_treat=find("treatment of wastewater, average", nf="treatment of wastewater, average"),
    psa_ref=find("biogas purification to biomethane by pressure swing adsorption", ["RoW", "GLO"] + LOC),
)
bg["ad_elec"] = bg["drying_heat"] = bg["elec"]             # AD and dryer run on grid electricity
# heat supply: Paper 1 configuration -> None (code path of Paper 1, results identical)
_src = {"electricity": "electricity", "ng": bg["ad_heat"], "own_biogas": "own_biogas"}
PAPER1_HEAT = dict(htc="electricity", dryer="electricity", ad="ng")
heat_supply = None if {**PAPER1_HEAT, **HEAT_SUPPLY} == PAPER1_HEAT else {c: _src[v] for c, v in HEAT_SUPPLY.items()}
heat_backup = bg["elec"] if HEAT_BACKUP == "electricity" else bg["ad_heat"]
print("Heat supply:", "Paper 1 (HTC and dryer electric, digester natural gas)" if heat_supply is None
      else f"{HEAT_SUPPLY}, backup {HEAT_BACKUP}")
heat_mix = bwl.resolve_heat_mix(ei, lcx.heat_mix_datasets(CTX), HEAT_MIX_SHARES.keys(), LOC)
# heat displaced by biomethane: Paper 1 = NG heat market; provinces = NG boilers and furnaces (no CHP)
_ng = lcx.ng_heat_datasets(CTX)
ng_heat_comp = bwl.resolve_heat_mix(ei, {"ng": _ng}, ["ng"], LOC)["ng"] if _ng else None
try:
    ash_disposal = bwl.find_exact(ei, "market for wood ash mixture, pure", LOC)
except LookupError:
    ash_disposal = None
    print("WARNING: ash disposal dataset not found, boiler ash omitted")
co2_air = bwl.get_flow("Carbon dioxide, non-fossil", ("air",), bwl.CORE_FLOW_LOCATIONS)
co_air = bwl.get_flow("Carbon monoxide, non-fossil", ("air",), bwl.CORE_FLOW_LOCATIONS)

print(pd.DataFrame([{"role": k, "dataset": a["name"], "location": a.get("location")}
                    for k, a in bg.items() if k not in ("ad_elec", "drying_heat")]).to_string(index=False))

## 3. Build and run the system

In [ ]:
# =============================================================================
# 3. BUILD AND RUN (one foreground database per pathway)
# =============================================================================
def foreground(P):
    name = f"Pathway{P}"
    if name not in bd.databases:
        bd.Database(name).register()
    db = bd.Database(name)
    g = lambda code, label, unit: sb.get_or_create(db, code, label, unit)
    fg = dict(
        htc_reactor=g("htc_reactor", "HTC reactor", "unit"),
        drying=g("hydrochar_drying", "Hydrochar drying", "unit"),
        heat_prod=g("heat_prod_hydrochar", "Heat production from hydrochar", "megajoule"),
        sub_heat_coal=g("substitute_heat_coal", "Hard coal heat", "megajoule"),
        ad_process=g("anaerobic_digestion", "Anaerobic digestion (ADM1)", "unit"),
        dewatering=g("digestate dewatering", "Digestate dewatering", "unit"),
        biogas_upgrading=g("biogas_upgrading", "Biogas upgrading (PSA)", "unit"),
        heat_biomethane=g("heat_biomethane", "Heat production from biomethane", "megajoule"),
        heat_naturalgas=g("heat_naturalgas", "Natural gas heat (substituted)", "megajoule"),
        qc_heat_mix=g("qc_industrial_heat_mix", "Industrial combustion heat mix (substituted)", "megajoule"),
        waste_treatment=g("waste_treatment_1kg", "Waste treatment of organic waste (HTC-AD system)", "kilogram"),
    )
    wt0 = fg["waste_treatment"]       # clear results stored by previous runs
    keep = {"name", "code", "unit", "database", "type", "location", "reference product", "id"}
    for k in [k for k in list(wt0._data) if k not in keep]:
        del wt0._data[k]
    wt0.save()
    return name, fg

lcc.clear_cache()
results, contributions = {}, {}
for P in PATHWAYS:
    FG, fg = foreground(P)
    ctx = sb.SystemContext(
        ei=ei, bios=bios, FOREGROUND_DB=FG, **bg, co2_air=co2_air, co_air=co_air, **fg,
        PATHWAY=P, m_feedstock=1000.0, default_params=feed,
        ml_params={**feed, "T": T_HTC, "RT": RT_HTC},
        T_HTC=T_HTC, RT_HTC=RT_HTC, target_moisture_HTC=TARGET_MOISTURE_HTC,
        MC_char=MC_CHAR, target_MC_char_dry=MC_CHAR_DRY,
        COD_conc_target=COD_TARGET, ISR_batch=ISR_BATCH,
        feedstock_name=FEEDSTOCK_TYPE, b_feed_overrides=B_FEED_OVERRIDES or None,
        pw_ch4_multiplier=PW_CH4_MULTIPLIER, heat_supply=heat_supply, heat_backup=heat_backup,
        heat_mix_components_qc=heat_mix, ng_heat_components=ng_heat_comp, ash_disposal=ash_disposal,
        **lcx.ctx_kwargs(CTX))
    if ADM1_MODE == "continuous":
        ss = sb.spinup_ss_state(ctx, pathway=P, adm1_HRT_days=HRT_DAYS, t_spinup=T_SPINUP,
                                TS_target=TS_TARGET, verbose=False, ml_module=ML_MODULE)
        wt, _, adm1_out, _, inv = sb.build_system(
            ctx, ml_module=ML_MODULE, adm1_mode="continuous", adm1_HRT_days=HRT_DAYS,
            adm1_ss_state=ss, ISR_batch=ISR_BATCH, TS_target=TS_TARGET)
    else:                                    # BMP test: no spin-up, blank-corrected
        wt, _, adm1_out, _, inv = sb.build_system(
            ctx, ml_module=ML_MODULE, adm1_mode="batch", adm1_t_sim_days=BATCH_DAYS,
            ISR_batch=ISR_BATCH, TS_target=TS_TARGET)

    MJ_hc = float(wt.get("MJ_hydrochar") or 0.0)
    scores, contrib = {}, {}
    for k, m in METHODS.items():
        lca = LCA({wt: 1}, m); lca.lci(); lca.lcia()
        s = lca.score
        c = lcc.contributions(wt, FG, m, total=s)
        if HYDROCHAR_DISPLACES == "coal":
            # same credit with hard coal heat instead of the heat mix (SI Section S6.2)
            d = MJ_hc * (lcc.unit_score(fg["qc_heat_mix"], m) - lcc.unit_score(bg["coal_heat"], m))
            c["Credit: industrial heat mix"] += d
            c = {("Credit: hard coal heat" if n == "Credit: industrial heat mix" else n): v for n, v in c.items()}
            s += d
        scores[k], contrib[k] = s, c
    results[P] = dict(scores=scores, wt=dict(wt._data), inv=inv,
                      pH=(adm1_out.get("ss_state") or {}).get("pH"))
    contributions[P] = contrib
    print(f"Pathway {P} done. QA flags: {wt.get('qa_flags') or 'none'}")

## 4. Results

In [ ]:
# =============================================================================
# 4. RESULTS per t of wet feedstock (negative impact = net benefit)
# =============================================================================
pd.set_option("display.width", 200, "display.max_columns", 20)

impacts = pd.DataFrame({f"Pathway {P}": r["scores"] for P, r in results.items()})
print(f"Impacts (hydrochar displaces {HYDROCHAR_DISPLACES})")
display(impacts.style.format("{:.4g}"))

PROCESS = {"Y_char (%)": ("inv", "hydrochar yield (% dry)"), "HHV_char (MJ/kg)": ("inv", "hydrochar HHV (MJ/kg)"),
           "HC_dry_yield": ("wt", "dried hydrochar (kg)"), "MJ_hydrochar": ("wt", "hydrochar heat (MJ)"),
           "biomethane_m3": ("wt", "biomethane (Nm3)"), "CH4_mass_AD": ("wt", "CH4 (kg)"),
           "E_HTC_elec_kWh": ("wt", "HTC electricity (kWh)"), "E_dryer_heat_kWh": ("wt", "dryer (kWh)"),
           "Q_AD_util_MJ": ("wt", "digester heat (MJ)"),
           "heat_demand_htc_MJ": ("wt", "heat demand HTC (MJ)"), "heat_demand_dryer_MJ": ("wt", "heat demand dryer (MJ)"),
           "heat_demand_ad_MJ": ("wt", "heat demand digester (MJ)"), "CH4_own_use_MJ": ("wt", "own biogas burned (MJ CH4)"),
           "heat_shortfall_htc_MJ": ("wt", "backup heat HTC (MJ)"), "heat_shortfall_dryer_MJ": ("wt", "backup heat dryer (MJ)"),
           "heat_shortfall_ad_MJ": ("wt", "backup heat digester (MJ)")}
proc = pd.DataFrame({f"Pathway {P}": {lab: r[src].get(k) for k, (src, lab) in PROCESS.items()} | {"digester pH": r["pH"]}
                     for P, r in results.items()})
print("Process results")
display(proc)

# what this run used (context, options) and every cap or QA check that bound
RUN_INFO = {"adm1_mode": "ADM1 mode", "location_code": "location", "heat_supply": "heat supply",
            "ng_heat_credit": "heat displaced by biomethane",
            "pw_ch4_multiplier": "A: PW methane multiplier", "pw_ch4_multiplier_effective": "A: multiplier after cap",
            "adm1_fD": "B: fD", "adm1_f_sol": "B: soluble share", "adm1_k_hyd": "B: k_hyd (1/d)",
            "caps_active": "caps that bound", "qa_flags": "QA flags"}
info = pd.DataFrame({f"Pathway {P}": {lab: r["wt"].get(k) for k, lab in RUN_INFO.items()} for P, r in results.items()})
info.loc["heat mix displaced"] = ", ".join(f"{k} {v:.1%}" for k, v in HEAT_MIX_SHARES.items())
info.loc["heat mix source"] = CTX["heat_mix_source"]
print("Run settings and checks")
with pd.option_context("display.max_colwidth", 120):
    display(info.fillna("–"))

for k in METHODS:
    print(f"Contributions: {k}")
    display(pd.DataFrame({f"Pathway {P}": c[k] for P, c in contributions.items()}).style.format("{:.4g}"))